### Das ägyptische Grabmal – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024), 1. Runde, Aufgabe 5*

Viele Computerspiele, Abenteuer- und Science-Fiction-Filme enthalten gefährliche Fallen, durch welche die Heldin oder der Held durch
geschicktes Hin- und Herlaufen hindurchkommen können.

Petra steht vor einem solchen Szenario: Ein Grabmal kann nur durch einen langen Gang erreicht werden. Dieser Gang besteht aus
Abschnitten, die lückenlos aufeinander folgen. In jedem Abschnitt ist in der Decke ein Steinquader eingelassen, der ihn blockieren kann.

Am Anfang ist jeder Abschnitt durch seinen Quader blockiert.

Beim Erreichen des Ganges löst Petra einen Mechanismus aus, der die Quader bewegt. Petra beobachtet, dass sich jeder Quader periodisch
bewegt mit einer ganzzahligen Anzahl von Minuten als Periodenlänge.

Ein Quader mit Periode 5 Minuten blockiert seinen Abschnitt zunächst 5 Minuten, um ihn dann für 5 Minuten freizugeben und nach 5
Minuten erneut zu blockieren.

Petra überlegt sich zunächst ein einfaches Beispiel: Drei Quader mit den Perioden 5, 8 und 12 Minuten durchlaufen fünf
Konfigurationen, wie im Bild dargestellt.

Man kann den Gang passieren, indem man nach 8 Minuten in den zweiten Abschnitt läuft, dort 4 Minuten wartet und dann durch den
dritten Abschnitt zum Grabmal läuft.

Petra kann sich nur in Abschnitten aufhalten, die nicht blockiert sind. Steht sie in einem Abschnitt, kann sie in einen benachbarten
Abschnitt nur dann laufen, wenn beide Abschnitte gleichzeitig nicht blockiert sind.

#### Aufgabe 5

Schreibe ein Programm, das Instruktionen für ein schnellstmögliches Erreichen des Grabmals ausgibt. Für die Eingabe 5, 8, 12 könnten
diese Instruktionen so aussehen: *Warte 8 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe zum Grabmal.*

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Das Bild auf dem Aufgabenblatt

Das Bild zeigt den Gang mit den drei Abschnitten (links der Eingang mit Petra, rechts das Grabmal) zu den Zeitpunkten 0, 5, 8, 10
und 12 Minuten:

| nach ... Minuten | Abschnitt 1 (Periode 5) | Abschnitt 2 (Periode 8) | Abschnitt 3 (Periode 12) |
|:-:|:-:|:-:|:-:|
| 0  | blockiert | blockiert | blockiert |
| 5  | frei      | blockiert | blockiert |
| 8  | frei      | frei      | blockiert |
| 10 | blockiert | frei      | blockiert |
| 12 | blockiert | frei      | frei      |

#### Beispieldaten

Im Ordner `beispieldaten` stehen `grabmal0.txt` bis `grabmal5.txt`. Die erste Zeile enthält die Anzahl `n` der Quader, dann folgt
für jeden Quader eine Zeile mit seiner Periode in Minuten (vom Eingang zum Grabmal):

```
3
5
8
12
```

`grabmal0.txt` ist das Beispiel aus der Aufgabenstellung. Für `grabmal5.txt` (85 Quader) braucht ein einfaches Programm viel Zeit.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Rest beim Teilen (Modulo):** Der Quader mit der Periode `p` blockiert seinen Abschnitt in der Minute `t`, wenn `t % (2 · p) < p`
  ist, und gibt ihn sonst frei. So kann man zu jeder Zeit nachrechnen, ob ein Abschnitt offen ist, ohne die Bewegung der Quader
  nachzuspielen.
- **Zustände mit der Zeit:** Petra ist zu einer Zeit `t` in einem Abschnitt `i`. Von dort kann sie **warten** (Zeit läuft weiter,
  wenn der Abschnitt offen bleibt) oder **laufen** (zum Nachbarabschnitt, ohne Zeit, wenn beide offen sind). Man sucht also einen Weg
  durch ein Raster aus „Abschnitt × Zeit“.
- **Mengen von möglichen Orten:** Statt einen einzelnen Weg zu verfolgen, merkt man sich in jeder Minute die **Menge aller Abschnitte**,
  in denen Petra sein könnte. Das ist einfach, aber bei langen Wartezeiten zu langsam.
- **Zeitfenster:** Ein Abschnitt ist in langen, zusammenhängenden Zeiträumen offen. Innerhalb eines solchen **Fensters** ändert sich
  nichts. Es kommt also nur darauf an, in welches Fenster man kommt und wann, nicht auf jede einzelne Minute.
- **Dijkstra:** Die Knoten sind (Abschnitt, Fenster), die „Länge“ eines Wegs ist die Zeit. Der Algorithmus von Dijkstra mit einer
  Prioritätswarteschlange findet die früheste Ankunft. Der Graph wird dabei nicht komplett aufgebaut, sondern Nachbarn werden erst
  berechnet, wenn man einen Knoten besucht.
- **Weg rekonstruieren:** Zu jedem besuchten Knoten merkt man sich den Vorgänger und läuft ihn vom Ziel aus zurück.

**Nützliche Funktionen und Bibliotheken**
- `heapq.heappush` und `heapq.heappop` für die Prioritätswarteschlange
- `dict` für die besuchten Fenster, Tupel `(abschnitt, fenster)` als Schlüssel
- `%` (Rest) und `//` (ganzzahlige Division)

## Die Idee

1. Die **Perioden einlesen** und eine Funktion bauen, die sagt, ob ein Abschnitt zu einer Zeit offen ist.
2. Zuerst die **einfachste Lösung**: Minute für Minute die Menge der Abschnitte verfolgen, in denen Petra sein kann, bis der letzte Abschnitt
   dabei ist. Sie liefert die Zeit, aber nur bei kleinen Beispielen schnell genug.
3. Dann die **schnelle Lösung**: Jeder Abschnitt hat Zeitfenster, in denen er offen ist. Mit Dijkstra über (Abschnitt, Fenster) findet man
   die früheste Ankunft und den Weg dorthin.
4. Aus dem Weg die **Anweisungen** für Petra machen („Warte …, laufe …“).
5. Die Anweisungen **selbst nachprüfen**: Nachspielen zeigt, ob jeder Schritt erlaubt ist und ob Petra zur richtigen Zeit am Ziel ist.

## Schritt 1: Die Eingabe lesen

Die Datei hat in der ersten Zeile die Zahl `n` der Quader, danach die `n` Perioden. Wir lesen alle Zahlen der Datei, die erste ist `n`.
Der Abschnitt mit der Nummer `i + 1` (in der Aufgabe zählt man ab 1) hat die Periode `perioden[i]`.

In [ ]:
def lies_gang(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = []
        for text in f.read().split():
            zahlen.append(int(text))
    n = zahlen[0]
    return zahlen[1:1 + n]


perioden = lies_gang('beispieldaten/grabmal0.txt')
print(perioden)

[5, 8, 12]


## Schritt 2: Wann ist ein Abschnitt offen?

Ein Quader mit der Periode `p` blockiert seinen Abschnitt in den Minuten `0` bis `p − 1`, gibt ihn in den Minuten `p` bis `2p − 1` frei,
blockiert ihn wieder von `2p` bis `3p − 1` und so weiter. Das ist genau die Rechnung `t % (2 · p)`: Ist der Rest kleiner als `p`, ist der Abschnitt
blockiert, sonst offen.

Wir zeichnen das für die Perioden `5, 8, 12` von `grabmal0.txt`: eine Zeile pro Minute, ein `#` für blockiert und ein `.` für offen. So sieht
man das Bild aus der Aufgabe für jede Minute. Das Grabmal ist nach 12 Minuten zu erreichen, weil dann zum ersten Mal ein Weg bis zum
letzten Abschnitt frei ist.

In [ ]:
def ist_offen(p, t):
    return t % (2 * p) >= p


print('Minute  Abschnitt 1 2 3')
for t in range(0, 15):
    zeile = str(t).rjust(6) + '  '
    for p in perioden:
        if ist_offen(p, t):
            zeile = zeile + ' .'
        else:
            zeile = zeile + ' #'
    print(zeile)

Minute  Abschnitt 1 2 3
     0   # # #
     1   # # #
     2   # # #
     3   # # #
     4   # # #
     5   . # #
     6   . # #
     7   . # #
     8   . . #
     9   . . #
    10   # . #
    11   # . #
    12   # . .
    13   # . .
    14   # . .


In Minute 8 sind die Abschnitte 1 und 2 offen (`. .`), der dritte noch nicht. In Minute 12 ist der Abschnitt 3 offen, der Abschnitt 2 noch,
und der Abschnitt 1 blockiert. Petra kann in Minute 8 durch den Abschnitt 1 in den Abschnitt 2 laufen, dort warten und in Minute 12
weiter in den Abschnitt 3.

## Schritt 3: Die einfachste Lösung, Minute für Minute

Wir merken uns in jeder Minute die **Menge der Abschnitte**, in denen Petra sein kann. In der Minute `t` geschieht Folgendes:

1. **Warten:** Petra kann in einem Abschnitt bleiben, in dem sie in der vorigen Minute war, wenn er auch jetzt offen ist.
2. **Vom Eingang:** Ist der erste Abschnitt offen, kann Petra vom Eingang aus hinein, denn sie darf warten, solange sie will.
3. **Laufen:** Von jedem Abschnitt, in dem sie jetzt sein kann, kommt sie in jeden **zusammenhängend offenen** Nachbarabschnitt, nach links
   und nach rechts, denn das Laufen kostet keine Zeit.

Sobald der letzte Abschnitt in der Menge ist, ist Petra am Grabmal. Die Funktion gibt die Minute zurück, oder `None`, wenn bis
`grenze` Minuten nichts passiert.

In [ ]:
def minuten_simulation(perioden, grenze):
    n = len(perioden)
    moegliche = set()                                      # Abschnitte (ab 0), in denen Petra sein kann
    for t in range(grenze):
        offen = []
        for p in perioden:
            offen.append(ist_offen(p, t))
        neue = set()
        for i in moegliche:
            if offen[i]:
                neue.add(i)                                # warten
        if offen[0]:
            neue.add(0)                                    # vom Eingang in den ersten Abschnitt
        for i in list(neue):                               # laufen, solange die Abschnitte offen sind
            j = i + 1
            while j < n and offen[j]:
                neue.add(j)
                j = j + 1
            j = i - 1
            while j >= 0 and offen[j]:
                neue.add(j)
                j = j - 1
        moegliche = neue
        if n - 1 in moegliche:
            return t
    return None


for nr in range(4):
    perioden = lies_gang('beispieldaten/grabmal' + str(nr) + '.txt')
    print('grabmal' + str(nr) + '.txt:', len(perioden), 'Quader, nach', minuten_simulation(perioden, 1000000), 'Minuten am Grabmal')

grabmal0.txt: 3 Quader, nach 12 Minuten am Grabmal
grabmal1.txt: 5 Quader, nach 49 Minuten am Grabmal
grabmal2.txt: 5 Quader, nach 490021 Minuten am Grabmal
grabmal3.txt: 10 Quader, nach 35 Minuten am Grabmal


Für `grabmal0.txt` kommt Petra nach 12 Minuten ans Ziel. Bei `grabmal1.txt` sind es 49, bei `grabmal2.txt` schon 490 021 und bei `grabmal3.txt` 35.

**Warum reicht das nicht?** Bei `grabmal5.txt` kommt Petra erst nach knapp 44 Millionen Minuten ans Ziel, und es gibt 85 Abschnitte. Das Programm
müsste fast vier Milliarden Mal einen Abschnitt prüfen. Außerdem sagt es nur **wann**, nicht **wie**. Wir brauchen die Anweisungen.
Dafür nutzen wir, dass sich lange Zeit gar nichts ändert.

## Schritt 4: Zeitfenster

Der Quader mit der Periode `p` gibt seinen Abschnitt in einem **Fenster** frei: Das `k`-te Fenster (ab `k = 0`) geht von der Minute
`(2k + 1) · p` bis zur Minute `(2k + 2) · p − 1`. Danach ist der Abschnitt `p` Minuten blockiert, dann beginnt das nächste Fenster. Mit
`p = 5` sind die Fenster zum Beispiel `5 … 9`, `15 … 19`, `25 … 29` und so weiter.

Das macht das Problem viel einfacher:

- Innerhalb eines Fensters kann Petra in dem Abschnitt **beliebig lange warten**, bis zur letzten Minute des Fensters.
- Von einem Abschnitt `i` (Fenster `[a, b]`) in den Nachbarabschnitt `j` kommt sie genau dann, wenn dessen Fenster `[a', b']` **sich mit
  `[a, b]` überschneidet** und sie zu einer Minute in beiden Fenstern ist. Frühestens ist das in der Minute `max(Ankunft, a')`.

Ein **Knoten** ist ein Paar `(abschnitt, fenster)`. Wir suchen für jeden Knoten die **früheste Minute**, in der Petra dort sein kann. Das ist
ein kürzester Weg, wenn man die Zeit als Länge nimmt, und dafür gibt es den Algorithmus von Dijkstra.

In [ ]:
def fenster(p, k):
    """Erste und letzte Minute des k-ten offenen Fensters eines Quaders mit Periode p."""
    return (2 * k + 1) * p, (2 * k + 2) * p - 1


for k in range(4):
    print('Periode 5, Fenster', k, ':', fenster(5, k))

Periode 5, Fenster 0 : (5, 9)
Periode 5, Fenster 1 : (15, 19)
Periode 5, Fenster 2 : (25, 29)
Periode 5, Fenster 3 : (35, 39)


## Schritt 5: Dijkstra über Abschnitt und Fenster

In der **Prioritätswarteschlange** liegen Einträge `(zeit, abschnitt, fenster, vorgaenger_abschnitt, vorgaenger_fenster)`. Der
kleinste Eintrag ist immer der mit der kleinsten Zeit. In `vorgaenger` merken wir für jeden besuchten Knoten die Zeit seiner Ankunft
und seinen Vorgänger.

Der Ablauf:

1. **Start:** Petra steht am Eingang und kann in jedem Fenster des ersten Abschnitts hineingehen. Wir legen zuerst das Fenster 0 ab. Sobald
   wir ein solches **Start-Fenster** besuchen, legen wir das nächste ab (der Vorgänger ist dann −1). So gibt es immer genug Startmöglichkeiten.
2. Wir nehmen den kleinsten Eintrag. Ist der Knoten schon besucht, überspringen wir ihn. Sonst besuchen wir ihn: Ist es der **letzte
   Abschnitt**, sind wir fertig, denn Dijkstra besucht Knoten nach wachsender Zeit.
3. Für beide **Nachbarabschnitte** gehen wir die Fenster durch, die sich mit dem aktuellen Fenster überschneiden und nicht schon vor der
   Ankunftszeit enden. Jedes legen wir mit der Zeit `max(Ankunft, Fensteranfang)` in die Warteschlange.
4. Am Ende gehen wir von dort mit den Vorgängern rückwärts zum Start. `weg` ist die Liste der Knoten `(abschnitt, fenster, zeit)` vom Start
   bis zum Ziel.

`besuchte` ist die Zahl der besuchten Knoten, an der man sieht, wie wenig gerechnet wird.

In [ ]:
from heapq import heappush, heappop


def schnellster_weg(perioden):
    """Ergebnis: (früheste Ankunft im letzten Abschnitt, Weg, Zahl der besuchten Knoten). Abschnitte werden ab 0 gezählt."""
    n = len(perioden)
    warteschlange = [(perioden[0], 0, 0, -1, -1)]          # Start: Fenster 0 des ersten Abschnitts, Ankunft in Minute p
    vorgaenger = {}
    while len(warteschlange) > 0:
        zeit, i, k, vi, vk = heappop(warteschlange)
        if (i, k) in vorgaenger:
            continue                                       # schon besucht, und zwar früher
        vorgaenger[(i, k)] = (vi, vk, zeit)
        if i == n - 1:                                     # der letzte Abschnitt: fertig
            weg = []
            knoten = (i, k)
            while knoten[0] != -1:
                vi2, vk2, zeit2 = vorgaenger[knoten]
                weg.append((knoten[0], knoten[1], zeit2))
                knoten = (vi2, vk2)
            weg.reverse()
            return zeit, weg, len(vorgaenger)
        if vi == -1:                                       # Start-Fenster: das nächste Start-Fenster vormerken
            anfang_naechstes, ende_naechstes = fenster(perioden[0], k + 1)
            heappush(warteschlange, (anfang_naechstes, 0, k + 1, -1, -1))
        anfang, ende = fenster(perioden[i], k)
        for j in [i - 1, i + 1]:
            if 0 <= j < n:
                p = perioden[j]
                k2 = max(0, zeit // (2 * p) - 1)
                while fenster(p, k2)[1] < zeit:            # Fenster überspringen, die schon vorbei sind
                    k2 = k2 + 1
                while fenster(p, k2)[0] <= ende:           # Fenster, die sich mit dem aktuellen überschneiden
                    if (j, k2) not in vorgaenger:
                        heappush(warteschlange, (max(zeit, fenster(p, k2)[0]), j, k2, i, k))
                    k2 = k2 + 1
    return None


perioden = lies_gang('beispieldaten/grabmal0.txt')
ankunft, weg, besuchte = schnellster_weg(perioden)
print('Ankunft im letzten Abschnitt in Minute', ankunft, ', besuchte Knoten:', besuchte)
print('Weg (Abschnitt ab 0, Fenster, Minute):', weg)

Ankunft im letzten Abschnitt in Minute 12 , besuchte Knoten: 3
Weg (Abschnitt ab 0, Fenster, Minute): [(0, 0, 5), (1, 0, 8), (2, 0, 12)]


Für `grabmal0.txt` ist der Weg `(0, 0, 5)`: Start im Fenster 0 des ersten Abschnitts (er öffnet sich in Minute 5), dann `(1, 0, 8)`: Abschnitt 2 in
Minute 8, schließlich `(2, 0, 12)`: der letzte Abschnitt in Minute 12.

## Schritt 6: Die Anweisungen

Aus dem Weg machen wir eine Liste von **Bewegungen** `(minute, abschnitt)`: Jeder Schritt von einem Knoten zum nächsten ist ein Lauf in einen
neuen Abschnitt, und zwar zur Minute, in der Petra den nächsten Knoten betritt. Der Lauf **vom Eingang in den ersten Abschnitt** kann
irgendwann im Start-Fenster geschehen. Wir legen ihn auf die Minute des nächsten Laufs, damit Petra nicht unnötig im ersten Abschnitt
wartet. Gibt es nur einen Abschnitt, ist es die Ankunftszeit.

Dann gehen wir die Bewegungen der Reihe nach durch:
- **Läufe zur selben Minute** fasst man zusammen: Es zählt nur das letzte Ziel (der Lauf durch mehrere Abschnitte).
- Liegt zwischen der letzten Bewegung und der nächsten Zeit, kommt vorher ein `warte`.

Das Ergebnis ist eine Liste von Paaren `('warte', minuten)` und `('laufe', abschnitt)`. Daraus bauen wir den Text. Bei dem letzten Abschnitt
heißt es „laufe zum Grabmal“.

In [ ]:
def anweisungen_aus_weg(weg):
    bewegungen = []
    for j in range(1, len(weg)):
        bewegungen.append((weg[j][2], weg[j][0] + 1))      # Abschnitte ab 1 zählen
    if len(bewegungen) > 0:
        erste_zeit = bewegungen[0][0]
    else:
        erste_zeit = weg[0][2]
    bewegungen.insert(0, (erste_zeit, 1))                  # vom Eingang in den ersten Abschnitt
    anweisungen = []
    jetzt = 0
    i = 0
    while i < len(bewegungen):
        zeit, ziel = bewegungen[i]
        while i + 1 < len(bewegungen) and bewegungen[i + 1][0] == zeit:
            i = i + 1
            ziel = bewegungen[i][1]                        # gleiche Minute: nur das letzte Ziel zählt
        if zeit > jetzt:
            anweisungen.append(('warte', zeit - jetzt))
            jetzt = zeit
        anweisungen.append(('laufe', ziel))
        i = i + 1
    return anweisungen


def anweisungen_text(anweisungen, n):
    teile = []
    for art, wert in anweisungen:
        if art == 'warte':
            if wert == 1:
                teile.append('warte 1 Minute')
            else:
                teile.append('warte ' + str(wert) + ' Minuten')
        elif wert == n:
            teile.append('laufe zum Grabmal')
        else:
            teile.append('laufe in den Abschnitt ' + str(wert))
    text = ', '.join(teile) + '.'
    return text[0].upper() + text[1:]


anweisungen = anweisungen_aus_weg(weg)
print(anweisungen)
print(anweisungen_text(anweisungen, len(perioden)))

[('warte', 8), ('laufe', 2), ('warte', 4), ('laufe', 3)]
Warte 8 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe zum Grabmal.


Für `grabmal0.txt` kommt genau die Anweisung aus der Aufgabe heraus: *Warte 8 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe zum
Grabmal.* Beachte, dass das Programm auch beim Warten im ersten Abschnitt nichts verlangt, was nicht geht: Petra wartet am Eingang, bis sie
in einem Zug durch Abschnitt 1 in Abschnitt 2 laufen kann.

## Schritt 7: Die Anweisungen nachprüfen

Wir spielen die Anweisungen nach, ohne die Suche zu benutzen. Petra beginnt am Eingang (Ort 0, Minute 0).

- Bei `warte x`: Ist Petra in einem Abschnitt, muss er in dieser Zeit durchgehend offen sein. Beide Minuten (die jetzige und die nach dem
  Warten) liegen dann **im selben Fenster**. Am Eingang darf sie beliebig warten.
- Bei `laufe y`: Alle Abschnitte auf dem Weg, bei einem Lauf vom Eingang also die Abschnitte 1 bis `y`, müssen in dieser Minute offen sein.

Am Ende muss Petra im letzten Abschnitt sein, und die Zeit ist die Ankunftszeit. Dazu prüfen wir bei den kleinen Beispielen, ob die
Minuten-Simulation aus Schritt 3 **dieselbe Zeit** findet. Das zeigt, dass Dijkstra nichts Früheres übersehen hat.

In [ ]:
def im_selben_fenster(p, t1, t2):
    return ist_offen(p, t1) and ist_offen(p, t2) and (t1 - p) // (2 * p) == (t2 - p) // (2 * p)


def spiele_nach(perioden, anweisungen):
    """Spielt die Anweisungen nach. Ergebnis: die Minute der Ankunft im letzten Abschnitt oder None, wenn etwas nicht erlaubt ist."""
    ort = 0                                                # 0 ist der Eingang, sonst die Nummer des Abschnitts
    zeit = 0
    for art, wert in anweisungen:
        if art == 'warte':
            if ort >= 1 and not im_selben_fenster(perioden[ort - 1], zeit, zeit + wert):
                return None
            zeit = zeit + wert
        else:
            if ort == 0:
                von = 1
            else:
                von = ort
            bis = wert
            erster = min(von, bis)
            letzter = max(von, bis)
            for abschnitt in range(erster, letzter + 1):
                if not ist_offen(perioden[abschnitt - 1], zeit):
                    return None
            ort = wert
    if ort == len(perioden):
        return zeit
    return None


perioden = lies_gang('beispieldaten/grabmal1.txt')
ankunft, weg, besuchte = schnellster_weg(perioden)
anweisungen = anweisungen_aus_weg(weg)
print('grabmal1: Ankunft', ankunft, ', nachgespielt:', spiele_nach(perioden, anweisungen))
print(anweisungen_text(anweisungen, len(perioden)))

grabmal1: Ankunft 49 , nachgespielt: 49
Warte 17 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe in den Abschnitt 3, warte 6 Minuten, laufe in den Abschnitt 4, warte 8 Minuten, laufe in den Abschnitt 3, warte 4 Minuten, laufe in den Abschnitt 2, warte 10 Minuten, laufe zum Grabmal.


## Schritt 8: Alle Beispiele

Für jedes Beispiel geben wir die Ankunftszeit, die Zahl der besuchten Knoten und das Ergebnis des Nachspielens aus. Bei den kleinen
Beispielen vergleichen wir zusätzlich mit der Minuten-Simulation. Die Anweisungen drucken wir für `grabmal0` bis `grabmal4` vollständig, bei
`grabmal5` ist es eine lange Liste, dort zeigen wir Anfang und Ende.

In [ ]:
ergebnisse = {}
for nr in range(6):
    perioden = lies_gang('beispieldaten/grabmal' + str(nr) + '.txt')
    ankunft, weg, besuchte = schnellster_weg(perioden)
    anweisungen = anweisungen_aus_weg(weg)
    nachgespielt = spiele_nach(perioden, anweisungen)
    ergebnisse[nr] = (perioden, ankunft, anweisungen)
    text = 'grabmal' + str(nr) + '.txt: ' + str(len(perioden)).rjust(2) + ' Quader, Ankunft nach ' + str(ankunft).rjust(8) + ' Minuten, '
    text = text + str(besuchte).rjust(6) + ' Knoten besucht, nachgespielt: ' + str(nachgespielt == ankunft)
    if nr <= 3:
        text = text + ', Minuten-Simulation: ' + str(minuten_simulation(perioden, 1000000) == ankunft)
    print(text)

grabmal0.txt:  3 Quader, Ankunft nach       12 Minuten,      3 Knoten besucht, nachgespielt: True, Minuten-Simulation: True
grabmal1.txt:  5 Quader, Ankunft nach       49 Minuten,      9 Knoten besucht, nachgespielt: True, Minuten-Simulation: True
grabmal2.txt:  5 Quader, Ankunft nach   490021 Minuten,      9 Knoten besucht, nachgespielt: True, Minuten-Simulation: True
grabmal3.txt: 10 Quader, Ankunft nach       35 Minuten,     23 Knoten besucht, nachgespielt: True, Minuten-Simulation: True
grabmal4.txt: 10 Quader, Ankunft nach  3432897 Minuten,   1785 Knoten besucht, nachgespielt: True
grabmal5.txt: 85 Quader, Ankunft nach 43965990 Minuten, 103004 Knoten besucht, nachgespielt: True


**Ergebnis:** Die Ankunftszeiten stimmen mit den offiziellen Lösungshinweisen überein:

| Beispiel | Quader | benötigte Minuten |
|---|---|---|
| grabmal0 | 3 | 12 |
| grabmal1 | 5 | 49 |
| grabmal2 | 5 | 490 021 |
| grabmal3 | 10 | 35 |
| grabmal4 | 10 | 3 432 897 |
| grabmal5 | 85 | 43 965 990 |

Das Programm betrachtet nicht jede Minute einzeln. Bei `grabmal2` besucht es nur 9 Knoten, obwohl 490 021 Minuten vergehen, bei `grabmal4`
sind es 1785 Knoten für 3,4 Millionen Minuten. Selbst bei `grabmal5` mit 85 Abschnitten sind es nur gut 100 000 Knoten, statt knapp 44 Millionen
Minuten mal 85 Abschnitte.

Die Anweisungen. Es gibt oft mehrere gleich schnelle Wege, deshalb sehen sie vielleicht anders aus als in den Lösungshinweisen. Die
Gesamtzeit ist immer dieselbe.

In [ ]:
for nr in range(5):
    perioden, ankunft, anweisungen = ergebnisse[nr]
    print('grabmal' + str(nr) + '.txt (' + str(ankunft) + ' Minuten):')
    print(anweisungen_text(anweisungen, len(perioden)))
    print()
perioden, ankunft, anweisungen = ergebnisse[5]
print('grabmal5.txt (' + str(ankunft) + ' Minuten), ' + str(len(anweisungen)) + ' Anweisungen. Die ersten und letzten:')
print(anweisungen_text(anweisungen[:10], len(perioden)))
print('...')
print(anweisungen_text(anweisungen[-8:], len(perioden)))

grabmal0.txt (12 Minuten):
Warte 8 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe zum Grabmal.

grabmal1.txt (49 Minuten):
Warte 17 Minuten, laufe in den Abschnitt 2, warte 4 Minuten, laufe in den Abschnitt 3, warte 6 Minuten, laufe in den Abschnitt 4, warte 8 Minuten, laufe in den Abschnitt 3, warte 4 Minuten, laufe in den Abschnitt 2, warte 10 Minuten, laufe zum Grabmal.

grabmal2.txt (490021 Minuten):
Warte 170000 Minuten, laufe in den Abschnitt 2, warte 40009 Minuten, laufe in den Abschnitt 3, warte 59991 Minuten, laufe in den Abschnitt 4, warte 80015 Minuten, laufe in den Abschnitt 3, warte 39985 Minuten, laufe in den Abschnitt 2, warte 100021 Minuten, laufe zum Grabmal.

grabmal3.txt (35 Minuten):
Warte 22 Minuten, laufe in den Abschnitt 6, warte 13 Minuten, laufe zum Grabmal.

grabmal4.txt (3432897 Minuten):
Warte 3303195 Minuten, laufe in den Abschnitt 2, warte 360 Minuten, laufe in den Abschnitt 5, warte 37695 Minuten, laufe in den Abschnitt 6, warte 51407 Minuten, 

## Für Neugierige

**Gibt es immer eine Lösung?** Ja. Betrachte die Minute `T − 1` mit `T = 2 · p1 · p2 · … · pn`. Für jeden Abschnitt `j` ist `T − 1` bei
der Rechnung `% (2 · pj)` gleich `2 · pj − 1`, und das ist mindestens `pj`. Also sind **alle** Abschnitte offen, und Petra könnte
durchlaufen. Die einfachste Anweisung wäre „Warte `T − 1` Minuten und laufe zum Grabmal“, aber das ist nicht das Schnellste. Kleiner reicht
sogar das kleinste gemeinsame Vielfache aller Perioden, doppelt genommen.

**Laufzeit:** Die Minuten-Simulation braucht für jede der `t` Minuten Zeit proportional zu `n`, zusammen also `O(n · t)`. Bei
`grabmal5` sind das fast 4 Milliarden Schritte. Mit Dijkstra über Zeitfenster besuchen wir nur die Fenster, die auf dem Weg
liegen könnten, und die Zahl der Fenster eines Abschnitts mit Periode `p` bis zur Zeit `t` ist `t / (2p)`. Dazu kommt der Faktor
`log` für die Prioritätswarteschlange.

**Andere Anforderungen an die Anweisungen:** Wir haben nur die **früheste Ankunft** verlangt. Man könnte auch möglichst **wenige Anweisungen**
wollen. Dann verschmilzt man aufeinanderfolgende `warte`-Anweisungen, wenn Petra stattdessen länger an einem Ort warten kann. Oder man
beschränkt Petras Beweglichkeit auf einen Schritt pro Minute. Dann muss man bei jedem Schritt eine Minute weiterzählen, und die
Lösbarkeit ist nicht mehr garantiert.

**Der Gang als Graph:** Wären die Abschnitte nicht in einer Reihe, sondern in einem beliebigen Netz angeordnet, könnte man dasselbe Verfahren
nehmen. Man ersetzt „linker und rechter Nachbar“ durch die Nachbarn im Netz.

**MiniZinc:** Hier haben wir keinen Abschnitt, denn die Ankunftszeiten gehen bis 44 Millionen Minuten. Es ist ein Kürzeste-Wege-Problem, und
das löst Dijkstra viel besser als ein Solver mit einer Variablen pro Minute.